# AIS SFP Distance Prediction Training

Train an image-based position predictor from the Hugging Face `vision_offset_dataset`.
Data is expected under `ws_aic/data/distance_prediction/SFP`, and checkpoints are saved under `ws_aic/model`.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/src/ais/ais_distance_prediction')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR

PosixPath('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/src/ais/ais_distance_prediction')

In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from model import (
    VisionOffsetDataset,
    build_resnet_position_model,
    compute_target_stats,
    download_vision_offset_dataset,
    evaluate,
    fit,
    load_samples,
    seed_everything,
    split_samples_by_episode,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
DATASET_ROOT = Path('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/distance_prediction/SFP')
WEIGHT_ROOT = Path('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/model')

CAMERAS = ('left', 'center', 'right')
TARGET_KEYS = ('x_mm', 'y_mm', 'z_mm')
IMAGE_SIZE = (256, 288)
BATCH_SIZE = 32
NUM_WORKERS = 8
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'resnet50'
PRETRAINED = True
AGGREGATION = 'concat'
NUM_PORT_HEADS = 2
EXPAND_ALL_PORTS = False
VAL_EVERY_N = 5
MAX_TRAIN_BATCHES = None  # Set to None for full training.
MAX_VAL_BATCHES = None  # Set to None for full validation.
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 0.05
EARLY_STOPPING_MONITOR = 'euclidean'
RUN_NAME = f"sfp_distance_{BACKBONE}_{'_'.join(CAMERAS)}_{AGGREGATION}"

DATASET_ROOT, WEIGHT_ROOT / RUN_NAME

(PosixPath('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/distance_prediction/SFP'),
 PosixPath('/home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/model/sfp_distance_resnet50_left_center_right_concat'))

In [4]:
from collections import Counter

def split_samples_by_grid_holdout(samples, *, every_n=5):
    train, val = [], []
    for sample in samples:
        row = dict(sample)
        step_index = int(row.get('step_index', 0))
        if step_index % every_n == 0:
            val.append(row)
        else:
            train.append(row)
    return train, val, []


if not (DATASET_ROOT / 'samples.jsonl').is_file():
    raise FileNotFoundError(f'Missing SFP samples.jsonl: {DATASET_ROOT / "samples.jsonl"}')

samples = load_samples(DATASET_ROOT)
missing_images = [
    DATASET_ROOT / sample['images'][camera]
    for sample in samples
    for camera in CAMERAS
    if not (DATASET_ROOT / sample['images'][camera]).is_file()
]
if missing_images:
    preview = '\n'.join(str(path) for path in missing_images[:5])
    raise FileNotFoundError(f'{len(missing_images)} image files are missing. First missing files:\n{preview}')

train_samples, val_samples, test_samples = split_samples_by_grid_holdout(
    samples,
    every_n=VAL_EVERY_N,
)
target_stats = compute_target_stats(
    train_samples,
    TARGET_KEYS,
    expand_all_ports=EXPAND_ALL_PORTS,
)

print(f'dataset_root: {DATASET_ROOT}')
print(f'raw samples: total={len(samples)} train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}')
print(f'effective rows: train={len(train_samples) * (NUM_PORT_HEADS if EXPAND_ALL_PORTS else 1)} '
      f'val={len(val_samples) * (NUM_PORT_HEADS if EXPAND_ALL_PORTS else 1)}')
print(f'images checked: {len(samples) * len(CAMERAS)}')
print('ports:', dict(Counter(sample.get('port_name', 'unknown') for sample in samples)))
print('label frame example:', samples[0]['label']['frame'])
print('target mean:', target_stats['mean'].tolist())
print('target std :', target_stats['std'].tolist())

dataset_root: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/data/distance_prediction/SFP
raw samples: total=10000 train=8000 val=2000 test=0
effective rows: train=8000 val=2000
images checked: 30000
ports: {'sfp_port_0': 5000, 'sfp_port_1': 5000}
label frame example: task_board/nic_card_mount_0/sfp_port_0_link_entrance
target mean: [-0.19905997812747955, -0.3419622480869293, -20.417387008666992]
target std : [6.551285743713379, 7.665936470031738, 11.511677742004395]


In [5]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
train_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=train_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=train_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    expand_all_ports=EXPAND_ALL_PORTS,
)
val_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=val_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    transform=eval_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    expand_all_ports=EXPAND_ALL_PORTS,
)

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
    persistent_workers=NUM_WORKERS > 0,
)

batch = next(iter(train_loader))
batch['image'].shape, batch['target'].shape, batch['port_id'][:8], batch['raw_target'][:3]

/home/vsc/LLM_TUNE/structure-stability/venv/venv-stability/lib/python3.10/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


(torch.Size([32, 3, 3, 256, 288]),
 torch.Size([32, 3]),
 tensor([1, 0, 1, 0, 1, 1, 0, 0]),
 tensor([[ -4.7487,   0.5859, -38.2989],
         [  4.3804, -11.0259, -14.8430],
         [ -1.3019,   1.5642, -32.5315]]))

In [7]:
model = build_resnet_position_model(
    backbone_name=BACKBONE,
    pretrained=PRETRAINED,
    output_dim=len(TARGET_KEYS),
    hidden_dim=256,
    dropout=0.2,
    aggregation=AGGREGATION,
    num_views=len(CAMERAS),
    num_port_heads=NUM_PORT_HEADS,
)
model.to(device)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

sum(p.numel() for p in model.parameters() if p.requires_grad)


26720838

In [8]:
config = {
    'dataset_root': str(DATASET_ROOT),
    'cameras': CAMERAS,
    'target_keys': TARGET_KEYS,
    'target_mean': target_stats['mean'].tolist(),
    'target_std': target_stats['std'].tolist(),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'backbone': BACKBONE,
    'pretrained': PRETRAINED,
    'aggregation': AGGREGATION,
    'num_views': len(CAMERAS),
    'num_port_heads': NUM_PORT_HEADS,
    'expand_all_ports': EXPAND_ALL_PORTS,
    'max_train_batches': MAX_TRAIN_BATCHES,
    'max_val_batches': MAX_VAL_BATCHES,
    'early_stopping': {
        'monitor': EARLY_STOPPING_MONITOR,
        'mode': 'min',
        'patience': EARLY_STOPPING_PATIENCE,
        'min_delta': EARLY_STOPPING_MIN_DELTA,
    },
    'label_frame_mode': samples[0]['label'].get('frame_mode'),
    'label_coordinate': samples[0]['label'].get('coordinate'),
    'split': {'name': 'grid_holdout', 'val_every_n': VAL_EVERY_N},
}

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=EPOCHS,
    weight_dir=WEIGHT_ROOT,
    run_name=RUN_NAME,
    scheduler=scheduler,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    early_stopping_monitor=EARLY_STOPPING_MONITOR,
    max_train_batches=MAX_TRAIN_BATCHES,
    max_val_batches=MAX_VAL_BATCHES,
    config=config,
)
print(f'best checkpoint: {WEIGHT_ROOT / RUN_NAME / "best.pt"}')
history[-1]

epochs:   0%|          | 0/100 [00:00<?, ?it/s]

train 1/100:   0%|          | 0/250 [00:00<?, ?it/s]

val 1/100:   0%|          | 0/63 [00:00<?, ?it/s]

epoch=001 lr=1.00e-04 train_loss=0.0732 val_loss=0.0078 val_mae=0.823 val_euclidean=1.667


train 2/100:   0%|          | 0/250 [00:00<?, ?it/s]

...

epoch=050 lr=1.56e-06 train_loss=0.0100 val_loss=0.0003 val_mae=0.164 val_euclidean=0.332


train 51/100:   0%|          | 0/250 [00:00<?, ?it/s]

val 51/100:   0%|          | 0/63 [00:00<?, ?it/s]

epoch=051 lr=1.56e-06 train_loss=0.0098 val_loss=0.0005 val_mae=0.204 val_euclidean=0.417
early stopping at epoch=051; best_epoch=041 best_euclidean=0.3418
best checkpoint: /home/vsc/LLM_TUNE/AIC_Sejong/ws_aic/model/sfp_distance_resnet50_left_center_right_concat/best.pt


{'epoch': 51.0,
 'lr': 1.5625e-06,
 'lr_before_scheduler': 1.5625e-06,
 'scheduler_metric': 0.41705822944641113,
 'train_loss': 0.009774765292182564,
 'val_loss': 0.000468260039575398,
 'val_mae': 0.20361021161079407,
 'val_rmse': 0.27874842286109924,
 'val_euclidean': 0.41705822944641113}

In [9]:
val_metrics = evaluate(
    model,
    val_loader,
    device,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_metrics

val:   0%|          | 0/63 [00:00<?, ?it/s]

{'loss': 0.000468260039575398,
 'mae': 0.20361021161079407,
 'rmse': 0.27874842286109924,
 'euclidean': 0.41705822944641113}

In [10]:
model.eval()
batch = next(iter(val_loader))
with torch.inference_mode():
    pred = model(batch['image'].to(device)).cpu()
pred_mm = pred * target_stats['std'] + target_stats['mean']
preview = torch.cat([pred_mm[:8], batch['raw_target'][:8]], dim=1)
print('columns: pred_x pred_y pred_z target_x target_y target_z')
preview

columns: pred_x pred_y pred_z target_x target_y target_z


tensor([[  5.8393,  -2.1096, -34.6063,   5.9777,  -1.9683, -34.7300],
        [ -6.5812,   0.9519,  -3.0933,  -6.5667,   1.0901,  -2.6410],
        [  5.1558,  11.6752, -12.9439,   5.2435,  12.0123, -13.1644],
        [  6.3410,  -3.4123, -11.7409,   6.6247,  -3.3804, -11.6632],
        [  3.5098,  -1.1550, -22.8121,   3.5685,  -1.1473, -22.8374],
        [ -5.1531,  -5.6795, -26.8905,  -5.2504,  -5.8070, -26.7669],
        [ -8.1262,   4.7047, -18.3282,  -8.2620,   4.8542, -18.0555],
        [  4.3956,  -1.8658, -11.2280,   4.6382,  -1.7646, -11.0359]])